In [ ]:
# Install gwexpy with pinned versions of core dependencies for reproducibility on Colab

%pip install -q "gwexpy[all]" "gwpy<5.0.0" "numpy<2.0.0" "scipy<1.13.0" "astropy<7.0.0"

# セグメント解析: 基本パイプライン

**目的**: `SegmentTable` を使用して時刻キー付きデータ解析を管理する方法を学びます。

`SegmentTable` は、特定の時間セグメントに関連付けられたメタデータとペイロードデータ（TimeSeries や PSD など）のコンテナです。大規模なデータセットを効率的に扱うため、遅延読み込みをサポートしています。

## 1. SegmentTable の作成

GPS 開始・終了時刻を持つセグメントを定義した CSV ファイルからサンプルデータを読み込みます。

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

from pathlib import Path

from gwexpy.table import SegmentTable

# repo root 実行でも notebook ディレクトリ実行でも sample CSV を見つける。
csv_candidates = []
for root in [Path.cwd(), *Path.cwd().parents]:
    csv_candidates.extend([
        root / "docs" / "_static" / "samples" / "sample_segment_data.csv",
        root / "_static" / "samples" / "sample_segment_data.csv",
    ])

sample_csv = next((path for path in csv_candidates if path.exists()), None)
if sample_csv is None:
    tried = [str(path) for path in csv_candidates]
    raise FileNotFoundError(f"Could not find sample_segment_data.csv. Tried: {tried}")

st = SegmentTable.read(str(sample_csv))

print(st)
st.display().head()

## 2. セグメントの可視化

セグメントのタイムラインを素早く可視化できます。

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
st.segments(ax=ax, label="Tutorial Segment")
plt.title("SegmentTable Timeline")
plt.show()

## 3. ペイロードの遅延読み込み

`SegmentTable` ではカラムに「ローダー」を紐付けることができます。データは実際にアクセスされたときのみ読み込まれます。

In [ ]:
from gwexpy.noise.wave import gaussian


def noise_loader(segment):
    # セグメント用の合成ノイズを生成
    duration = float(segment[1] - segment[0])
    return gaussian(duration=duration, sample_rate=1024, t0=float(segment[0]))

# 注意: 遅延読み込み可能なペイロードデータには add_series_column を使用（kind='timeseries' など）
st.add_series_column("noise", loader=noise_loader, kind="timeseries")

# 最初の行のノイズにアクセス（読み込みが実行される）
data_0 = st.row(0)["noise"]
print(f"GPS {data_0.t0.value} から {len(data_0)} サンプルを読み込みました")

## 4. 行ごとの処理

行をイテレートしたり、`apply` を使ってデータを処理したりできます。

In [ ]:
# 各ノイズセグメントの RMS を計算
# 軽量なメタデータ結果には add_column を使用
st.add_column("rms", data=[row["noise"].rms().value for row in st])
st.display()

## 5. 検証セル（NBMAKE）

In [ ]:
assert "noise" in st.columns
assert len(st) > 0
print("検証完了！")